## Phase 2c: Cross-Training-Pair Generalization

Phase 2b showed that a language-conditioned query encoder closes 97.6% of the FLORES hard-pool gap for EN-ES training without collapsing bystander language retrieval. Phase 2c asks whether this holds across different training pairs.

Three training pairs: EN-FR, EN-DE, EN-SW. For each, we train vanilla InfoNCE and Phase 2b (lambda=0.5) to 2K steps, then evaluate on the same 5-language FLORES hard pool. The target language in the hard pool changes per training pair — for EN-FR we want the model to retrieve French; for EN-DE, German; for EN-SW, Swahili.

**Predictions:**
- EN-SW should show almost no hard-pool gap — Swahili is typologically isolated with no close Romance or Germanic competitors in the pool
- EN-FR and EN-DE should show gaps concentrated on closely related neighbors
- Phase 2b should close the gap for each pair without collapsing retrieval in the other languages

In [ ]:
!pip install -q transformers datasets torch

import os, math, random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from transformers import (XLMRobertaModel, XLMRobertaTokenizerFast,
                          get_linear_schedule_with_warmup)
from datasets import load_dataset
import matplotlib.pyplot as plt
import numpy as np

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

MAX_STEPS    = 2000
BATCH_SIZE   = 32
LR           = 2e-5
TEMPERATURE  = 0.07
WARMUP_STEPS = 200
LOG_EVERY    = 200
LAMBDA_AUX   = 0.5

TRAIN_PAIRS = ['fr', 'de', 'sw']

LANG_ORDER       = ['es', 'fr', 'de', 'sw', 'ar']
LANG_TO_IDX      = {l: i for i, l in enumerate(LANG_ORDER)}
NUM_LANGS        = len(LANG_ORDER)
LANG_CODES       = {'es': 'spa_Latn', 'fr': 'fra_Latn', 'de': 'deu_Latn',
                    'sw': 'swh_Latn', 'ar': 'arb_Arab'}
LANG_HARD_OFFSET = {'es': 0, 'fr': 1, 'de': 2, 'sw': 3, 'ar': 4}

CROSS_WEIGHTS = {'es': 1.0, 'fr': 2.0, 'de': 2.0, 'sw': 1.0, 'ar': 1.0}

# EN-ES Phase 1 / 2b results for comparison
ENES_VANILLA_2K = {'easy': 0.9941, 'target': 0.3824, 'any': 1.0000, 'gap': 0.6117}
ENES_PHASE2B_2K = {'easy': 0.9951, 'target': 0.9802, 'any': 0.9980, 'gap': 0.0149}
ENES_BYSTANDER  = {'es': 0.9951, 'fr': 0.9960, 'de': 0.9990, 'sw': 0.9476, 'ar': 0.9733}

tokenizer = XLMRobertaTokenizerFast.from_pretrained('xlm-roberta-base')
print('Setup complete.')

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## Model Definitions

In [ ]:
class ProjectionHead(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=2048, output_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, output_dim))
    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)


class XLMRWrapper(nn.Module):
    def __init__(self, model_name='xlm-roberta-base'):
        super().__init__()
        self.model      = XLMRobertaModel.from_pretrained(model_name)
        self.projection = ProjectionHead(768, 2048, 256)

    def forward(self, input_ids, attention_mask):
        out = self.model(input_ids=input_ids, attention_mask=attention_mask)
        return self.projection(self._mean_pool(out.last_hidden_state, attention_mask))

    def encode(self, input_ids, attention_mask):
        out = self.model(input_ids=input_ids, attention_mask=attention_mask)
        return self._mean_pool(out.last_hidden_state, attention_mask)

    @staticmethod
    def _mean_pool(hidden, mask):
        m = mask.unsqueeze(-1).float()
        return F.normalize((hidden * m).sum(1) / m.sum(1).clamp(min=1e-9), dim=-1)


class LanguageConditionedWrapper(nn.Module):
    def __init__(self, model_name='xlm-roberta-base'):
        super().__init__()
        self.model      = XLMRobertaModel.from_pretrained(model_name)
        self.projection = ProjectionHead(768, 2048, 256)
        self.lang_emb   = nn.Embedding(NUM_LANGS, 768)
        nn.init.normal_(self.lang_emb.weight, std=0.01)

    def _mean_pool_unnorm(self, hidden, mask):
        m = mask.unsqueeze(-1).float()
        return (hidden * m).sum(1) / m.sum(1).clamp(min=1e-9)

    def encode_base(self, input_ids, attention_mask):
        out = self.model(input_ids=input_ids, attention_mask=attention_mask)
        return F.normalize(
            self._mean_pool_unnorm(out.last_hidden_state, attention_mask), dim=-1)

    def forward(self, input_ids, attention_mask, lang=None):
        base = self.encode_base(input_ids, attention_mask)
        if lang is not None:
            idx  = torch.tensor(LANG_TO_IDX[lang], device=input_ids.device)
            base = base + self.lang_emb(idx)
        return self.projection(F.normalize(base, dim=-1))

    def encode(self, input_ids, attention_mask, lang=None):
        base = self.encode_base(input_ids, attention_mask)
        if lang is not None:
            idx  = torch.tensor(LANG_TO_IDX[lang], device=input_ids.device)
            base = base + self.lang_emb(idx)
        return F.normalize(base, dim=-1)


print('Model definitions ready.')

## Loss Functions

In [ ]:
def infonce_loss(z_src, z_tgt, tau=TEMPERATURE):
    sim    = z_src @ z_tgt.T / tau
    labels = torch.arange(sim.size(0), device=sim.device)
    return (F.cross_entropy(sim, labels) + F.cross_entropy(sim.T, labels)) / 2.0


def aux_lang_cond_loss(model, flores_encs, tau=TEMPERATURE):
    en_base = model.encode_base(
        flores_encs['en']['input_ids'],
        flores_encs['en']['attention_mask'])

    z_cands = {
        lang: model.forward(
            flores_encs[lang]['input_ids'],
            flores_encs[lang]['attention_mask'],
            lang=None)
        for lang in LANG_ORDER}

    total_loss = 0.0
    for tgt in LANG_ORDER:
        lang_idx = torch.tensor(LANG_TO_IDX[tgt], device=DEVICE)
        z_query  = model.projection(
            F.normalize(en_base + model.lang_emb(lang_idx), dim=-1))

        pos        = (z_query * z_cands[tgt]).sum(-1) / tau
        all_scores = z_query @ z_cands[tgt].T / tau

        for other in LANG_ORDER:
            if other == tgt:
                continue
            w    = CROSS_WEIGHTS[other]
            hard = (z_query * z_cands[other]).sum(-1, keepdim=True) / tau
            if w != 1.0:
                hard = hard + math.log(w)
            all_scores = torch.cat([all_scores, hard], dim=1)

        total_loss += -(pos - torch.logsumexp(all_scores, dim=1)).mean()

    return total_loss / NUM_LANGS


print('Loss functions ready.')

## Utilities and Data

In [ ]:
def collate_pairs(pairs, max_length=128):
    src = tokenizer([p[0] for p in pairs], padding=True, truncation=True,
                    max_length=max_length, return_tensors='pt')
    tgt = tokenizer([p[1] for p in pairs], padding=True, truncation=True,
                    max_length=max_length, return_tensors='pt')
    return src, tgt


def collate_sixtuples(batch, max_length=128):
    return {
        lang: tokenizer(
            [row[lang] for row in batch], padding=True, truncation=True,
            max_length=max_length, return_tensors='pt')
        for lang in ['en'] + LANG_ORDER}


def opus_stream(target_lang, seed=42):
    """Stream (EN, target_lang) pairs. Tries en-XX then XX-en config."""
    try:
        ds = load_dataset('opus100', f'en-{target_lang}', split='train', streaming=True)
    except Exception:
        ds = load_dataset('opus100', f'{target_lang}-en', split='train', streaming=True)
    ds = ds.shuffle(buffer_size=10000, seed=seed)
    for ex in ds:
        yield ex['translation']['en'], ex['translation'][target_lang]


def flores_dev_stream(dev_en, dev_sents, batch_size=32, seed=0):
    rng     = random.Random(seed)
    indices = list(range(len(dev_en)))
    while True:
        rng.shuffle(indices)
        buf = []
        for i in indices:
            buf.append({lang: (dev_en if lang == 'en' else dev_sents[lang])[i]
                        for lang in ['en'] + LANG_ORDER})
            if len(buf) == batch_size:
                yield buf
                buf = []


print('Utilities ready.')

In [ ]:
def load_flores(lang_code, split='devtest'):
    ds = load_dataset('openlanguagedata/flores_plus', lang_code, split=split)
    return [ex['text'] for ex in ds]

print('Loading FLORES dev (auxiliary training signal)...')
dev_en    = load_flores('eng_Latn', split='dev')
dev_sents = {lang: load_flores(code, split='dev') for lang, code in LANG_CODES.items()}
print(f'  {len(dev_en)} sentences per language')

print('Loading FLORES devtest (evaluation)...')
en_sents   = load_flores('eng_Latn', split='devtest')
lang_sents = {lang: load_flores(code, split='devtest') for lang, code in LANG_CODES.items()}
N = len(en_sents)
print(f'  {N} sentences per language  |  hard pool = {N * NUM_LANGS}')

## Evaluation Functions

In [ ]:
@torch.no_grad()
def _encode_all(mdl, target_lang=None):
    """Encode FLORES devtest. EN conditioned on target_lang if model supports it."""
    def enc(sents, lang=None):
        out = []
        for i in range(0, len(sents), 256):
            tok = {k: v.to(DEVICE) for k, v in tokenizer(
                sents[i:i+256], padding=True, truncation=True,
                max_length=128, return_tensors='pt').items()}
            if lang is not None and hasattr(mdl, 'lang_emb'):
                embs = mdl.encode(tok['input_ids'], tok['attention_mask'], lang=lang)
            else:
                embs = mdl.encode(tok['input_ids'], tok['attention_mask'])
            out.append(embs.cpu())
        return torch.cat(out, 0)

    en    = enc(en_sents, lang=target_lang)
    langs = {lang: enc(lang_sents[lang]) for lang in LANG_ORDER}
    return en, langs


def evaluate(mdl, target_lang):
    """Hard-pool P@1. EN queries conditioned on target_lang for Phase 2b models."""
    mdl.eval()
    en_embs, lang_embs = _encode_all(mdl, target_lang)

    offset       = LANG_HARD_OFFSET[target_lang]
    correct_hard = torch.arange(N) + offset * N

    p1_easy   = float((en_embs @ lang_embs[target_lang].T).argmax(1)
                      .eq(torch.arange(N)).float().mean())
    hard_pool = torch.cat([lang_embs[l] for l in LANG_ORDER], dim=0)
    top1      = (en_embs @ hard_pool.T).argmax(dim=1)
    p1_target = float(top1.eq(correct_hard).float().mean())
    p1_any    = float((top1 % N).eq(torch.arange(N)).float().mean())

    return p1_easy, p1_target, p1_any


def bystander_eval(mdl):
    """For each language, condition EN queries on that language and measure easy-pool P@1."""
    mdl.eval()
    results = {}
    for lang in LANG_ORDER:
        en_embs, lang_embs = _encode_all(mdl, lang)
        top1 = (en_embs @ lang_embs[lang].T).argmax(dim=1)
        results[lang] = float(top1.eq(torch.arange(N)).float().mean())
    return results


print('Eval functions ready.')

## Training Functions

In [ ]:
def train_vanilla(target_lang):
    ckpt_path = f'vanilla_infonce_2000steps_en{target_lang}.pt'

    if os.path.exists(ckpt_path):
        print(f'  {ckpt_path} found, loading...')
        ckpt  = torch.load(ckpt_path, map_location=DEVICE)
        model = XLMRWrapper().to(DEVICE)
        model.load_state_dict(ckpt['model_state_dict'])
        p1_easy, p1_target, p1_any = evaluate(model, target_lang)
        print(f'  easy={p1_easy:.4f}  target={p1_target:.4f}  any={p1_any:.4f}')
        del model; torch.cuda.empty_cache()
        return p1_easy, p1_target, p1_any

    print(f'Training vanilla EN-{target_lang.upper()} to {MAX_STEPS} steps...')
    model     = XLMRWrapper().to(DEVICE)
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    scheduler = get_linear_schedule_with_warmup(optimizer, WARMUP_STEPS, MAX_STEPS)

    model.train()
    train_iter = opus_stream(target_lang, seed=42)
    buf, step  = [], 0

    for src_text, tgt_text in train_iter:
        buf.append((src_text, tgt_text))
        if len(buf) < BATCH_SIZE:
            continue
        src_enc, tgt_enc = collate_pairs(buf); buf = []
        src_enc = {k: v.to(DEVICE) for k, v in src_enc.items()}
        tgt_enc = {k: v.to(DEVICE) for k, v in tgt_enc.items()}
        z_src = model(src_enc['input_ids'], src_enc['attention_mask'])
        z_tgt = model(tgt_enc['input_ids'], tgt_enc['attention_mask'])
        loss  = infonce_loss(z_src, z_tgt)
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        step += 1
        if step % LOG_EVERY == 0:
            print(f'  step {step}/{MAX_STEPS}  loss={loss.item():.4f}')
        if step >= MAX_STEPS:
            break

    p1_easy, p1_target, p1_any = evaluate(model, target_lang)
    torch.save({'model_state_dict': model.state_dict(), 'step': step}, ckpt_path)
    print(f'  easy={p1_easy:.4f}  target={p1_target:.4f}  any={p1_any:.4f}')
    print(f'  saved {ckpt_path}  ({os.path.getsize(ckpt_path)/1e6:.0f} MB)')
    del model, optimizer, scheduler; torch.cuda.empty_cache()
    return p1_easy, p1_target, p1_any


def train_phase2b(target_lang):
    ckpt_path = f'lang_cond_l{LAMBDA_AUX}_2000steps_en{target_lang}.pt'

    if os.path.exists(ckpt_path):
        print(f'  {ckpt_path} found, loading...')
        ckpt  = torch.load(ckpt_path, map_location=DEVICE)
        model = LanguageConditionedWrapper().to(DEVICE)
        model.load_state_dict(ckpt['model_state_dict'])
        p1_easy, p1_target, p1_any = evaluate(model, target_lang)
        print(f'  easy={p1_easy:.4f}  target={p1_target:.4f}  any={p1_any:.4f}')
        del model; torch.cuda.empty_cache()
        return p1_easy, p1_target, p1_any

    print(f'Training Phase 2b EN-{target_lang.upper()} (lambda={LAMBDA_AUX}) to {MAX_STEPS} steps...')
    model     = LanguageConditionedWrapper().to(DEVICE)
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    scheduler = get_linear_schedule_with_warmup(optimizer, WARMUP_STEPS, MAX_STEPS)

    model.train()
    opus_iter   = opus_stream(target_lang, seed=42)
    flores_iter = flores_dev_stream(dev_en, dev_sents, batch_size=BATCH_SIZE, seed=0)
    buf, step   = [], 0

    for src_text, tgt_text in opus_iter:
        buf.append((src_text, tgt_text))
        if len(buf) < BATCH_SIZE:
            continue
        src_enc, tgt_enc = collate_pairs(buf); buf = []
        src_enc = {k: v.to(DEVICE) for k, v in src_enc.items()}
        tgt_enc = {k: v.to(DEVICE) for k, v in tgt_enc.items()}
        z_src     = model.forward(src_enc['input_ids'], src_enc['attention_mask'], lang=target_lang)
        z_tgt     = model.forward(tgt_enc['input_ids'], tgt_enc['attention_mask'], lang=None)
        loss_main = infonce_loss(z_src, z_tgt)

        flores_batch = next(flores_iter)
        flores_encs  = {
            lang: {k: v.to(DEVICE) for k, v in enc.items()}
            for lang, enc in collate_sixtuples(flores_batch).items()}
        loss_aux = aux_lang_cond_loss(model, flores_encs)

        loss = loss_main + LAMBDA_AUX * loss_aux
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        step += 1
        if step % LOG_EVERY == 0:
            print(f'  step {step}/{MAX_STEPS}  main={loss_main.item():.4f}  aux={loss_aux.item():.4f}')
        if step >= MAX_STEPS:
            break

    p1_easy, p1_target, p1_any = evaluate(model, target_lang)
    torch.save({'model_state_dict': model.state_dict(), 'step': step,
                'lambda_aux': LAMBDA_AUX, 'target_lang': target_lang}, ckpt_path)
    print(f'  easy={p1_easy:.4f}  target={p1_target:.4f}  any={p1_any:.4f}')
    print(f'  saved {ckpt_path}  ({os.path.getsize(ckpt_path)/1e6:.0f} MB)')
    del model, optimizer, scheduler; torch.cuda.empty_cache()
    return p1_easy, p1_target, p1_any


print('Training functions ready.')

In [ ]:
results_vanilla = {}
results_phase2b = {}

for tgt in TRAIN_PAIRS:
    print(f'\n{"="*52}')
    print(f'  Training pair: EN-{tgt.upper()}')
    print(f'{"="*52}')

    print('\n[Vanilla InfoNCE]')
    e, t, a = train_vanilla(tgt)
    results_vanilla[tgt] = {'easy': e, 'target': t, 'any': a, 'gap': e - t}

    print('\n[Phase 2b]')
    e, t, a = train_phase2b(tgt)
    results_phase2b[tgt] = {'easy': e, 'target': t, 'any': a, 'gap': e - t}

print('\nAll training runs complete.')

In [ ]:
van_gap_enes = ENES_VANILLA_2K['gap']

print('=== Hard Pool P@1 at 2K Steps ===\n')
print(f'{"Pair":<8}  {"Model":<14}  {"P@1-easy":>9}  {"P@1-target":>11}  '
      f'{"P@1-any":>9}  {"Gap":>8}  {"Gap closed":>11}')
print('-' * 80)

print(f'{"EN-ES":<8}  {"Vanilla 2K":<14}  {ENES_VANILLA_2K["easy"]:>9.4f}  '
      f'{ENES_VANILLA_2K["target"]:>11.4f}  {ENES_VANILLA_2K["any"]:>9.4f}  '
      f'{ENES_VANILLA_2K["gap"]:>8.4f}  {"baseline":>11}')
print(f'{"EN-ES":<8}  {"Phase 2b":<14}  {ENES_PHASE2B_2K["easy"]:>9.4f}  '
      f'{ENES_PHASE2B_2K["target"]:>11.4f}  {ENES_PHASE2B_2K["any"]:>9.4f}  '
      f'{ENES_PHASE2B_2K["gap"]:>8.4f}  {"97.6%":>11}')
print()

for tgt in TRAIN_PAIRS:
    van  = results_vanilla[tgt]
    p2b  = results_phase2b[tgt]
    pct  = (van['gap'] - p2b['gap']) / van['gap'] * 100 if van['gap'] > 0 else 0.0
    pair = f'EN-{tgt.upper()}'
    print(f'{pair:<8}  {"Vanilla 2K":<14}  {van["easy"]:>9.4f}  {van["target"]:>11.4f}  '
          f'{van["any"]:>9.4f}  {van["gap"]:>8.4f}  {"baseline":>11}')
    print(f'{pair:<8}  {"Phase 2b":<14}  {p2b["easy"]:>9.4f}  {p2b["target"]:>11.4f}  '
          f'{p2b["any"]:>9.4f}  {p2b["gap"]:>8.4f}  {pct:>10.1f}%')
    print()

In [ ]:
print('=== Bystander Easy-Pool Eval (Phase 2b, EN conditioned per language) ===\n')
header = f'{"Pair":<10}' + ''.join(f'  {l.upper():>8}' for l in LANG_ORDER)
print(header)
print('-' * len(header))

print(f'{"EN-ES":<10}' + ''.join(f'  {ENES_BYSTANDER[l]:>8.4f}' for l in LANG_ORDER))

for tgt in TRAIN_PAIRS:
    ckpt_path = f'lang_cond_l{LAMBDA_AUX}_2000steps_en{tgt}.pt'
    ckpt  = torch.load(ckpt_path, map_location=DEVICE)
    model = LanguageConditionedWrapper().to(DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    bs = bystander_eval(model)
    del model; torch.cuda.empty_cache()
    pair = f'EN-{tgt.upper()}'
    print(f'{pair:<10}' + ''.join(f'  {bs[l]:>8.4f}' for l in LANG_ORDER))

print('\nDownloading checkpoints...')
from google.colab import files
for tgt in TRAIN_PAIRS:
    for prefix in [f'vanilla_infonce_2000steps_en{tgt}',
                   f'lang_cond_l{LAMBDA_AUX}_2000steps_en{tgt}']:
        path = f'{prefix}.pt'
        if os.path.exists(path):
            files.download(path)
            print(f'  Downloaded {path}')